# Day 2 · S5 — Build a gold standard

*Day 2 — Linguistic Data Analysis II*

**Day 2 has two notebooks, one per hands-on session** — S5 builds a gold standard by hand (this one), S6 measures a model against one (`day2-s6_evaluation_metrics.ipynb`). Submit both at the end of the day.

### How to use this notebook

You only edit the cells marked **✏️ YOU EDIT**. Cells marked **🔧 Library cell** are pre-written — run them, don't change them.

➡️ Work top to bottom. When you're done, **Runtime → Run all**, then **File → Download → Download `.ipynb`** and submit that file.

## Build a gold standard yourself

So far the gold labels have been handed to you. Now you make some. The work is split across **three surfaces that share one spine**, the six steps **A–F**:

- **Slides** — the *concepts*: how to sample (A), why annotate blind (C), how to read a confusion matrix (D–E).
- **A Google Sheet** — the *human judgment*: you and your partner annotate (C) and re-annotate (E) there.
- **This notebook** — the *numbers*: it reads the sheet, measures, adjudicates and exports (**D–F**).

So **steps A–C need no code**; this notebook first runs at **step D**. Each header below prints the same `A–F` label as the slides — find your place by the *letter*.

::: {.callout-note}
## Why a spreadsheet?
Annotation is a *judgement* task, not a coding task, and a sheet is what most annotation projects actually use. The point is to feel how often two careful people disagree, and what it takes to resolve that into a single defensible label.
:::

### A · Sample → copy your track's sheet   *(E&K Step 3 · ①)*

The sample is **already drawn for you** — one template Sheet per track. Its first tab is **`round1`**, with the columns **`ID · Text · CoderA · CoderB · Final · Note`** and only `ID` and `Text` filled in, never the labels, so your annotation is genuinely independent. *How* it was sampled — represent the domain, fix the unit, a seed for reproducibility — is the slide concept; the exact draw is in the reference appendix at the end.

**Open your track's template Sheet → `File → Make a copy`** into your own Drive. Then take your copy's **id** from its URL — the long string between `/d/` and `/edit`. You'll paste it into step D. (Opening by id, not name, means two copies both called "Copy of …" are never confused.)

### B · Apply the operationalized scheme   *(E&K Steps 4–5; Fuoli · ②)*

Before you label, restate the **decidable rule** you're annotating against — the scheme your team drafted in the earlier sessions — and skim the guidelines and per-level examples. One label per unit; know your label set cold. → interpret this on **step B** (slides).

### C · Annotate blind, in pairs   *(E&K Step 6 · ③)*

**Entirely in the Sheet — no code**, in the **`round1`** tab. One of you fills **`CoderA`** and the other **`CoderB`**, *without looking at each other's column*. Leave `Final` blank. Use `Note` for anything you found hard to decide; those notes are your evidence in step E.

::: {.callout-important}
## Stop here and go annotate
Label all ~20 rows in **both** annotator columns before running the next cell. Come back to Colab when the sheet is filled in — the notebook picks up at **step D**.
:::

### D · Measure agreement   *(E&K Step 6 · ③)*   ✏️ YOU EDIT

Colab opens here — run the two helper cells below first. Then read your copied sheet back in and measure how much the two of you agreed: **percent agreement**, **Cohen's κ** (agreement corrected for chance), and an **annotator-vs-annotator confusion matrix**, whose off-diagonal cells show *which label pairs* you confuse.

::: {.callout-note collapse="true"}
## How to read what this prints — the interpretation (step D, slides)
Percent agreement flatters two coders who both lean on the same label — they agree a lot *by luck*. **Cohen's κ** strips that luck out, so trust the κ, not the percentage (recall S4: 80% raw agreement was only κ ≈ 0.52). Then find the *one off-diagonal cell* dragging κ down — that label pair is your to-do list for step E. You will code κ yourself from scratch in the S6 notebook.
:::

In [ ]:
#@title 📦 Setup — run me first { display-mode: "form" }
# Helper — you don't need to read this. Run it and move on.
import json, urllib.request
from sklearn.metrics import confusion_matrix
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt

# CEFR-SP gold set — the published labels you compare against in step F.
GOLD_URL = "https://raw.githubusercontent.com/egumasa/linguistic-data-analysis-II-2026/main/sources/resources/datasets/gold/cefr_sentences.json"
LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
print(f"Setup done. scikit-learn ready.")

In [ ]:
#@title 🔧 Library cell: Google Sheets annotation round-trip { display-mode: "form" }
# Helper — you don't need to read this. Run it and move on.
#   Google Sheets annotation round-trip

# Sheet column headers (the annotation template uses these exact names):
COL_ID, COL_TEXT = "ID", "Text"
COL_A, COL_B = "CoderA", "CoderB"
COL_FINAL, COL_NOTES = "Final", "Note"
ANNOTATION_HEADER = [COL_ID, COL_TEXT, COL_A, COL_B, COL_FINAL, COL_NOTES]

def _sheets_client():
    """Authorise gspread with your Google account (a pop-up asks for permission)."""
    from google.colab import auth
    import google.auth, gspread
    auth.authenticate_user()           # the pop-up: "let Colab use your Sheets"
    creds, _ = google.auth.default()   # the permission slip that pop-up produced
    return gspread.authorize(creds)    # a logged-in connection to Google Sheets

def create_annotation_sheet(title, items, labels):
    """Create a Sheet in YOUR Drive: one row per item, blank columns to label.
    `items` are {"id","text",...} dicts — any existing label is deliberately NOT
    copied across, so you annotate blind. Returns the sheet URL."""
    ### Step 1: make an empty spreadsheet in your own Drive ###
    sheet = _sheets_client().create(title)
    worksheet = sheet.sheet1
    worksheet.update_title("round1")   # first round lives in the 'round1' tab

    ### Step 2: one row per item — id and text filled in, label columns left blank ###
    rows = []
    for item in items:
        #                id            text          CoderA CoderB Final Note
        rows.append([item["id"], item["text"], "", "", "", ""])

    ### Step 3: write it all in one go, then pin the header row ###
    worksheet.update([ANNOTATION_HEADER] + rows)   # header first, then the data
    worksheet.freeze(rows=1)                       # header stays put as you scroll
    print(f"Created '{title}' with {len(rows)} rows in tab 'round1'.")
    print("Allowed labels:", ", ".join(labels))
    print("Open it:", sheet.url)
    return sheet.url

def load_annotation_sheet(sheet_id, worksheet="round1"):
    """Read one TAB of your annotation sheet back as a list of row dicts.
    `sheet_id` is the long id in the sheet's URL:
        docs.google.com/spreadsheets/d/<THIS PART>/edit
    Pasting the whole URL works too — either way opens the exact sheet, so two
    copies that share a name (\"Copy of ...\") are never confused.
    `worksheet` is the TAB name (a \"round\"): each round lives in its own tab, so
    re-annotating in round2 never overwrites round1 — the analysis stays reproducible."""
    ### Step 1: open the sheet — a pasted URL and a bare id both work ###
    client = _sheets_client()
    if str(sheet_id).startswith("http"):
        sheet = client.open_by_url(sheet_id)
    else:
        sheet = client.open_by_key(sheet_id)

    ### Step 2: find the tab (the "round") — and say which tabs exist if it is missing ###
    try:
        ws = sheet.worksheet(worksheet)
    except Exception:
        tabs = [w.title for w in sheet.worksheets()]   # what IS in this sheet
        raise ValueError(f"No tab named {worksheet!r}. Tabs in this sheet: {tabs}")

    ### Step 3: read every row as a dict keyed by the header names ###
    rows = ws.get_all_records()        # [{"ID": 1, "Text": "...", "CoderA": "B1", ...}, ...]
    print(f"Read {len(rows)} rows from tab '{worksheet}'.")
    return rows

def to_canonical(rows, labels, column=COL_FINAL):
    """Turn annotation rows into canonical gold: [{"id","text","label"}, ...].
    Blank rows are skipped; labels outside `labels` are reported, not silently kept."""
    ### Step 1: sort every row into one of three piles ###
    gold, blank, invalid = [], 0, []     # usable rows · not labelled yet · typos
    for row in rows:
        label = str(row.get(column, "")).strip()   # .strip() drops stray spaces
        if not label:
            blank += 1                    # nobody has filled this row in yet
        elif label not in labels:
            invalid.append((row.get(COL_ID), label))   # e.g. "b1" or "B11"
        else:
            gold.append({"id": int(row[COL_ID]), "text": str(row[COL_TEXT]), "label": label})

    ### Step 2: report all three counts, so nothing is dropped silently ###
    print(f"{len(gold)} usable · {blank} still blank · {len(invalid)} invalid")
    if invalid:
        print("  fix these in the sheet, then re-run:", invalid[:10])   # first 10
    return gold

def annotator_agreement(rows, a=COL_A, b=COL_B):
    """Percent agreement + Cohen's κ between the two annotator columns, PLUS an
    annotator-vs-annotator confusion matrix (the diagonal is where you agreed;
    off-diagonal cells show which label pairs the two of you confuse)."""
    from sklearn.metrics import cohen_kappa_score
    ### Step 1: keep only the rows where BOTH annotators actually chose a label ###
    pairs = [(str(r.get(a, "")).strip(), str(r.get(b, "")).strip()) for r in rows]
    pairs = [(x, y) for x, y in pairs if x and y]    # drop half-finished rows
    if not pairs:
        print("No rows where BOTH annotators have labelled. Nothing to compare yet.")
        return None

    ### Step 2: two metrics — raw agreement, and agreement corrected for chance ###
    a_labels = [x for x, _ in pairs]                 # annotator A's choices
    b_labels = [y for _, y in pairs]                 # annotator B's choices
    percent = sum(x == y for x, y in pairs) / len(pairs)   # how often you matched
    kappa = cohen_kappa_score(a_labels, b_labels)    # ...minus the luck
    print(f"{len(pairs)} doubly-annotated · agreement {percent:.1%} · Cohen's κ {kappa:.3f}")

    ### Step 3: draw WHICH labels you two confuse, not just how often ###
    # annotator-vs-annotator confusion matrix (mirrors the gold-vs-model evaluate()):
    labels = sorted(set(a_labels) | set(b_labels))   # every label either of you used
    cm = confusion_matrix(a_labels, b_labels, labels=labels)
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.xlabel("Annotator B"); plt.ylabel("Annotator A")   # diagonal = you agreed
    plt.title("Annotator-vs-annotator confusion matrix")
    plt.tight_layout(); plt.show()
    return {"n": len(pairs), "percent_agreement": percent, "kappa": kappa}

def disagreements(rows, a=COL_A, b=COL_B):
    """The rows your two annotators labelled differently — your adjudication list."""
    # keep a row only if both annotators labelled it AND they chose differently:
    out = [r for r in rows
           if str(r.get(a, "")).strip() and str(r.get(b, "")).strip()
           and str(r[a]).strip() != str(r[b]).strip()]
    print(f"{len(out)} rows to adjudicate. Agree on a `Final` label for each in the sheet.")
    return pd.DataFrame(out)

def compare_to_published(gold, published):
    """How often does YOUR final label match the published gold, item by item?"""
    ### Step 1: look up the published label for every id ###
    lookup = {item["id"]: item["label"] for item in published}   # id -> their label
    shared = [(g["label"], lookup[g["id"]]) for g in gold if g["id"] in lookup]
    if not shared:
        print("No overlapping ids — did you keep the ids the sheet gave you?")
        return None

    ### Step 2: count the matches, then show only the rows where you differ ###
    agree = sum(mine == theirs for mine, theirs in shared)
    print(f"{agree}/{len(shared)} match the published label ({agree / len(shared):.1%})")
    return pd.DataFrame([{"id": g["id"], "yours": g["label"], "published": lookup[g["id"]],
                          "text": g["text"]}
                         for g in gold if g["id"] in lookup
                         and g["label"] != lookup[g["id"]]])

In [ ]:
SHEET_ID = "1AbCdEf...paste_yours"   # ✏️ the id in YOUR copied sheet's URL
                                     #    (…/spreadsheets/d/THIS/edit) — the whole URL works too
ROUND    = "round1"                  # ✏️ which round's tab to analyze

rows = load_annotation_sheet(SHEET_ID, ROUND)   # read that tab back into Python
annotator_agreement(rows)            # % agreement, κ, and the confusion matrix

### E · Read the matrix → refine → re-annotate   *(E&K Step 6; Fuoli princ. 2 · ③)*

A low κ is a diagnosis of your **scheme**, not your annotating. `disagreements(rows)` lists every row the two of you saw differently — the items behind those off-diagonal cells.

::: {.callout-note collapse="true"}
## What to do with this list — the judgment (step E, slides)
For the label pair the matrix flagged, **refine the scheme / guidelines** until the ambiguity becomes decidable: add a rule, a boundary case, an example. Then re-annotate in a **fresh round tab** (below) and re-run **step D** to see κ move. Iterate until agreement is acceptable — Fuoli's principle 2 in action.
:::

In [ ]:
disagreements(rows)   # the rows you two labelled differently — your worklist

::: {.callout-important}
## Re-annotate in a fresh round tab, then re-run step D
Don't overwrite `round1`. In the Sheet, **right-click the `round1` tab → Duplicate**, rename the copy **`round2`**, and re-label the confused items *there*. Then set **`ROUND = "round2"`** in step D and re-run it. `round1` stays intact, so every run is reproducible and you can watch κ climb round over round. Repeat (round3, …) until κ is acceptable, then move to step F.
:::

### F · Adjudicate → gold   *(E&K Step 6 → feeds ④⑤)*   ✏️ YOU EDIT

The last disagreements don't refine away — you **decide** them. In your **latest round tab**, fill a single `Final` label for every row (where you already agreed, `Final` is that agreed label), then read it back and convert it to canonical form. `to_canonical` refuses anything that isn't one of your allowed labels, so typos surface here rather than silently corrupting your gold set.

In [ ]:
rows = load_annotation_sheet(SHEET_ID, ROUND)   # re-read your latest round, `Final` filled in
my_gold = to_canonical(rows, LEVELS)            # reads the `Final` column
my_gold[:3]                                     # peek at the first three items

**How does your gold compare with the published gold?** The CEFR-SP labels came from language-education professionals, keeping only sentences where two of them agreed. Differing from them is not simply *wrong* — Arase's own experts agreed exactly only 37.6% of the time — but each difference needs a look and a defensible reason. → interpret this on **step F** (slides).

In [ ]:
#@title 🔧 Library cell: load_gold { display-mode: "form" }
# Helper — you don't need to read this. Run it and move on.
#   load_gold(url_or_path) → gold

def load_gold(url_or_path):
    """Read the canonical gold JSON: [{'id','text','label'}, ...]."""
    if str(url_or_path).startswith("http"):                 # a web address?
        raw = urllib.request.urlopen(url_or_path).read().decode("utf-8")  # download it
        gold = json.loads(raw)                              # JSON text -> list of dicts
    else:                                                   # otherwise a file on disk
        gold = json.loads(open(url_or_path, encoding="utf-8").read())
    print(f"Loaded {len(gold)} items. First one:", gold[0])  # proof it worked
    return gold

In [ ]:
published = load_gold(GOLD_URL)      # the CEFR-SP labels, for comparison only
compare_to_published(my_gold, published)   # how often you two agree, item by item

**Save your gold set to your Drive** — it belongs in **your** Drive, not the course repo, and it becomes S6's yardstick. See [Housing your data in Google Drive](../resources/tools/google-drive-data.md).

In [ ]:
# ✏️ Uncomment in Colab to save:
# from google.colab import drive; drive.mount("/content/drive")
# with open("/content/drive/MyDrive/my_gold_day2.json", "w", encoding="utf-8") as f:
#     json.dump(my_gold, f, ensure_ascii=False, indent=2)
# print("saved", len(my_gold), "items")

::: {.callout-note collapse="true"}
## Reference — how the template sheets were built (you don't run this)

The template Sheet you copied in step A was generated once, ahead of the session, so the sample is fixed and reproducible: the `create_annotation_sheet` helper (loaded above) fed by a **seeded** random draw, one sheet per track (`cefr`, `cars50`, `raamove`, `l2_errors`). Shown so the sampling is transparent; you do **not** run it.

```python
import random

N_ITEMS = 20            # how many sentences to annotate
random.seed(42)         # fixed seed = the same sample every time it is drawn

to_annotate = random.sample(gold, N_ITEMS)   # `gold` = your track's labelled pool
# The sheet gets ids + text only — never the labels — so annotation stays blind:
create_annotation_sheet("lda2_day2_cefr", to_annotate, LEVELS)
```
:::

***
## ✅ Before you submit

1. **Runtime → Run all** and check every cell ran without error.
2. Your agreement numbers and the annotator-vs-annotator matrix are visible (step D), for the **last** round you ran.
3. `my_gold` printed a list of `{id, text, label}` records, and step F's comparison against the published gold ran (step F).
4. **File → Download → Download `.ipynb`** and upload **both** of today's Day-2 notebooks.